# D_Gold - Modelo estrella

Gold consume las tablas Silver registradas en Unity Catalog y construye
4 dimensiones y 1 tabla de hechos, listas para Power BI.


In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

calendar = spark.table(f"{SILVER_SCHEMA}.calendar")



def write_gold(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )


# Dimensión calendario
df_dim_calendar = (
    calendar
    .withColumn(
        "month_name",
        F.when(F.col("month") == 1, "Enero")
         .when(F.col("month") == 2, "Febrero")
         .when(F.col("month") == 3, "Marzo")
         .when(F.col("month") == 4, "Abril")
         .when(F.col("month") == 5, "Mayo")
         .when(F.col("month") == 6, "Junio")
         .when(F.col("month") == 7, "Julio")
         .when(F.col("month") == 8, "Agosto")
         .when(F.col("month") == 9, "Setiembre")
         .when(F.col("month") == 10, "Octubre")
         .when(F.col("month") == 11, "Noviembre")
         .when(F.col("month") == 12, "Diciembre")
    )
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
)
write_gold(df_dim_calendar, f"{GOLD_SCHEMA}.dim_calendar")


print("GOLD - tabla dimensional dim_calendar generado correctamente")
print("Dimensiones: dim_calendar")



GOLD - tabla dimensional dim_calendar generado correctamente
Dimensiones: dim_calendar


In [0]:
%sql
SELECT 'dim_calendar' AS tabla, COUNT(*) AS total_filas FROM workspace.gold.dim_calendar



tabla,total_filas
dim_calendar,731


In [0]:
%sql
select * from workspace.gold.dim_calendar limit 2

date,year,month,day,week,day_of_week,ingestion_timestamp,source_file,_rescued_data,month_name,date_key
2023-01-09,2023,1,9,2,0,2026-11-21T20:47:44.989Z,/Volumes/workspace/default/filestore/tables/raw/calendar.csv,null,Enero,20230109
2023-01-27,2023,1,27,4,4,2026-11-21T20:47:44.989Z,/Volumes/workspace/default/filestore/tables/raw/calendar.csv,null,Enero,20230127
